# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ubaidrees/flyrank-ml-tasks/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

Signal verdicts:

Staleness vs. decline rate: MIXED. Decline rate rises from 51.2% (n=20,655) to 61.1% (n=9,171) as staleness increases from <90 to 90-180 days, supporting the hypothesis — but reverses to 47.1% (n=174) in the 180+ day bucket. This is a real pattern, not noise (n=174 is large enough that the ~14-point swing exceeds expected sampling error). Likely explanation: content that's been stale the longest may have already bottomed out, so it can't "decline" further in the current window — worth treating the 180+ day threshold as a known blind spot in the rule, not proof staleness doesn't matter.
CTR by position vs. tier: CONFIRMED, after fixing a data-quality issue first. Raw check showed 1-3 position CTR averaging 2.71 (impossible — CTR can't exceed 1.0), traced to rows with impressions_90d = 1 producing fake 100% CTR values. After filtering to impressions_90d >= 500 (the same volume threshold the rule itself uses), CTR clearly drops with worse position — 20+ position median CTR (0.08) is roughly half of the 11-20 tier (0.17). Note: 1-3 and 4-10 tiers are nearly identical (0.20 vs 0.24), so position alone doesn't cleanly separate CTR at the top end — the rule is right to combine CTR with position rather than relying on position alone.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("https://raw.githubusercontent.com/Ubaidrees/flyrank-ml-tasks/main/data/raw/content_refresh_anonymized.csv")

# ---- Signal 1: Staleness vs. decline (behind stale_visible_page) ----
df['staleness_bucket'] = pd.cut(
    df['days_since_last_update'],
    bins=[-1, 90, 180, np.inf],
    labels=['<90 days', '90-180 days', '180+ days']
)
staleness_check = df.groupby('staleness_bucket').agg(
    n=('content_id', 'count'),
    pct_declining=('trend_direction', lambda x: (x == 'down').mean().round(3))
)
print("Signal 1 — Staleness vs. decline rate:")
print(staleness_check)

# ---- Signal 2: CTR vs. position (behind low_ctr_visible_page) ----
df['position_bucket'] = pd.cut(
    df['avg_position'],
    bins=[0, 3, 10, 20, np.inf],
    labels=['1-3', '4-10', '11-20', '20+']
)
ctr_check = df.groupby('position_bucket').agg(
    n=('content_id', 'count'),
    avg_ctr=('ctr', 'mean')
)
print("\nSignal 2 — CTR by position tier:")
print(ctr_check)

Signal 1 — Staleness vs. decline rate:
                      n  pct_declining
staleness_bucket                      
<90 days          20655          0.512
90-180 days        9171          0.611
180+ days           174          0.471

Signal 2 — CTR by position tier:
                     n   avg_ctr
position_bucket                 
1-3               1141  2.714303
4-10             11842  0.651045
11-20             7273  0.323443
20+               8539  0.211333


/tmp/ipykernel_1333/3025287620.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  staleness_check = df.groupby('staleness_bucket').agg(
/tmp/ipykernel_1333/3025287620.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ctr_check = df.groupby('position_bucket').agg(


In [3]:
print(df[df['position_bucket'] == '1-3']['ctr'].describe())
print("\nTop 5 highest CTR values in this bucket:")
print(df[df['position_bucket'] == '1-3'].nlargest(5, 'ctr')[['content_id', 'ctr', 'impressions_90d']])

count    1141.000000
mean        2.714303
std        10.698930
min         0.000000
25%         0.000000
50%         0.000000
75%         0.360000
max       100.000000
Name: ctr, dtype: float64

Top 5 highest CTR values in this bucket:
                 content_id    ctr  impressions_90d
240    content_006b16e7a2e7  100.0                1
4606   content_3f3576c295f5  100.0                1
13661  content_bf398aa7400e  100.0                1
18825  content_98458bafe297  100.0                1
22607  content_bc2c0c7243df  100.0                1


In [4]:
# Redo Signal 2 with a minimum-volume filter, matching the rule's own threshold
df_filtered = df[df['impressions_90d'] >= 500]
ctr_check_filtered = df_filtered.groupby('position_bucket').agg(
    n=('content_id', 'count'),
    median_ctr=('ctr', 'median'),
    mean_ctr=('ctr', 'mean')
)
print("Signal 2 (volume-filtered, impressions_90d >= 500) — CTR by position tier:")
print(ctr_check_filtered)

Signal 2 (volume-filtered, impressions_90d >= 500) — CTR by position tier:
                    n  median_ctr  mean_ctr
position_bucket                            
1-3               480        0.20  0.349500
4-10             7084        0.24  0.338152
11-20            4459        0.17  0.266629
20+              4703        0.08  0.134712


/tmp/ipykernel_1333/271431731.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ctr_check_filtered = df_filtered.groupby('position_bucket').agg(


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
import os

# The one rule: stale AND visible AND well-ranked AND underperforming CTR
flagged = df[
    (df['days_since_last_update'] >= 180) &
    (df['impressions_90d'] >= 500) &
    (df['avg_position'] > 0) & (df['avg_position'] <= 20) &
    (df['ctr'] < 0.5)
].copy()

# Score: staleness and CTR-gap weighted equally (0-1 normalized within the flagged set)
flagged['staleness_norm'] = (flagged['days_since_last_update'] - flagged['days_since_last_update'].min()) / \
                             (flagged['days_since_last_update'].max() - flagged['days_since_last_update'].min())
flagged['ctr_gap_norm'] = (0.5 - flagged['ctr']) / 0.5  # bigger gap below 0.5 = higher score

flagged['score'] = (0.5 * flagged['staleness_norm'] + 0.5 * flagged['ctr_gap_norm']).round(3)
flagged['reason_code'] = 'stale_page_ctr_gap'
flagged['action'] = 'review_for_refresh'

queue = flagged.sort_values('score', ascending=False)[
    ['content_id', 'client_id', 'score', 'reason_code', 'action',
     'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr']
]

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Flagged {len(queue)} of {len(df)} total pages ({len(queue)/len(df)*100:.1f}%)")
print("\nTop 10:")
print(queue.head(10))

Flagged 10 of 30000 total pages (0.0%)

Top 10:
                 content_id          client_id  score         reason_code  \
7452   content_72496874f806  client_4ec9599fc2  0.760  stale_page_ctr_gap   
26840  content_7f116ae1f6f5  client_9400f1b21c  0.580  stale_page_ctr_gap   
20837  content_928af3e22c80  client_7f2253d7e2  0.422  stale_page_ctr_gap   
16751  content_cf56e2e2e282  client_7f2253d7e2  0.397  stale_page_ctr_gap   
12045  content_c2d929d83eaa  client_7f2253d7e2  0.342  stale_page_ctr_gap   
11630  content_6226ee6adc91  client_d029fa3a95  0.320  stale_page_ctr_gap   
26799  content_77d4d5930e5e  client_7f2253d7e2  0.307  stale_page_ctr_gap   
22872  content_e3ff1b093148  client_d029fa3a95  0.220  stale_page_ctr_gap   
5327   content_fe16a55cd13d  client_7f2253d7e2  0.217  stale_page_ctr_gap   
21268  content_0a91db491d14  client_7f2253d7e2  0.052  stale_page_ctr_gap   

                   action  days_since_last_update  impressions_90d  \
7452   review_for_refresh         

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [5]:
top10_review = """
TOP-10 REVIEW

1. content_72496874f806 (score 0.760) — Action: review_for_refresh.
   Why: extremely stale (301 days), real visibility (821 impressions), strong position (5.8), CTR 0.24 below target.
   Wrong if: this page is intentionally untouched because it's stable/evergreen — refreshing a working page risks breaking it, not fixing it.

2. content_7f116ae1f6f5 (score 0.580) — Action: review_for_refresh.
   Why: also 301 days stale, position 9.0.
   Wrong if: CTR (0.42) is actually ABOVE the 4-10 position tier's median (0.24) — this page isn't really underperforming on clicks, it just cleared the rule's absolute <0.5 cutoff. Weak pick.

3. content_928af3e22c80 (score 0.422) — Action: review_for_refresh.
   Why: 193 days stale, CTR (0.12) well below its 11-20 tier median (0.17), weak position (15.8).
   Wrong if: this query is highly competitive or seasonal, where low CTR is structural, not a content problem.

4. content_cf56e2e2e282 (score 0.397) — Action: review_for_refresh.
   Why: by far the highest volume in the queue (61,678 impressions), low CTR (0.15) at position 19.7 — largest potential upside if fixed.
   Wrong if: position 19.7 is borderline page-2; the real fix here may be a ranking problem, not a content refresh.

5. content_c2d929d83eaa (score 0.342) — Action: review_for_refresh.
   Why: strong volume (7,558), stale (193 days), CTR (0.20) close to its tier median (0.17).
   Wrong if: CTR is basically in line with its tier — the only real justification here is staleness, a weaker case than #1-4.

6. content_6226ee6adc91 (score 0.320) — Action: review_for_refresh.
   Why: stale (183 days), CTR (0.18) near tier norm.
   Wrong if: impressions_90d (545) barely clears the 500 threshold — a slightly different sampling window could drop this row out of the queue entirely.

7. content_77d4d5930e5e (score 0.307) — Action: review_for_refresh.
   Why: stale (194 days), good position (7.8), CTR (0.24) right at its tier median.
   Wrong if: CTR matches the tier norm exactly — this may just be a stale page with no real click problem, only an age problem.

8. content_e3ff1b093148 (score 0.220) — Action: review_for_refresh.
   Why: strong position (7.8), low volume (1,408).
   Wrong if: CTR (0.28) is close to or above its tier median — the "low CTR" trigger is weak here, likely flagged mostly on staleness.

9. content_fe16a55cd13d (score 0.217) — Action: review_for_refresh.
   Why: decent volume (4,556), stale (194 days).
   Wrong if: CTR (0.33) is actually well above its 11-20 tier median (0.17) — this may be a genuinely good performer only flagged by the absolute cutoff, not a real underperformer.

10. content_0a91db491d14 (score 0.052, lowest in queue) — Action: review_for_refresh.
    Why: high volume (13,299 impressions).
    Wrong if: CTR (0.49) is nearly double its 4-10 tier's median (0.24) and sits right at the rule's 0.5 cutoff edge — this is the weakest, most marginal flag in the whole top 10.
"""
print(top10_review)


TOP-10 REVIEW

1. content_72496874f806 (score 0.760) — Action: review_for_refresh.
   Why: extremely stale (301 days), real visibility (821 impressions), strong position (5.8), CTR 0.24 below target.
   Wrong if: this page is intentionally untouched because it's stable/evergreen — refreshing a working page risks breaking it, not fixing it.

2. content_7f116ae1f6f5 (score 0.580) — Action: review_for_refresh.
   Why: also 301 days stale, position 9.0.
   Wrong if: CTR (0.42) is actually ABOVE the 4-10 position tier's median (0.24) — this page isn't really underperforming on clicks, it just cleared the rule's absolute <0.5 cutoff. Weak pick.

3. content_928af3e22c80 (score 0.422) — Action: review_for_refresh.
   Why: 193 days stale, CTR (0.12) well below its 11-20 tier median (0.17), weak position (15.8).
   Wrong if: this query is highly competitive or seasonal, where low CTR is structural, not a content problem.

4. content_cf56e2e2e282 (score 0.397) — Action: review_for_refresh.
   Wh

## 4. Weak picks + leakage check
Weak picks: rows 2, 9, and 10 all have CTR above their own position tier's median (0.42 vs 0.24 tier median; 0.33 vs 0.17; 0.49 vs 0.24) — they only qualify because the rule uses an absolute ctr < 0.5 cutoff rather than a peer-relative one. A stronger version of this rule would compare CTR against the tier median, not a fixed number.

Client concentration: client_7f2253d7e2 appears in 6 of the top 10 rows. Not yet confirmed whether this reflects genuinely worse content from that client or an artifact of score normalization being computed across the whole flagged set (a client with unusually high-volume pages, like row 4's 61,678 impressions, could skew the shared normalization). Worth a follow-up check before trusting the ranking as client-fair.

Staleness caution: rows 1 and 2 (the two highest scores) both sit at 301 days stale — squarely in the 180+ day bucket that Signal 1 found has a reversed, weaker decline relationship (47.1% vs 61.1% in the 90-180 day bucket). The score may be overweighting staleness for exactly these two rows.

Leakage check: confirmed — the rule uses only days_since_last_update, impressions_90d, avg_position, and ctr, none of which are FlyRank product flags or future-window fields. trend_direction was used only for signal verification, never as a rule input.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.